<img src="https://raw.githubusercontent.com/stuartbowyer/sdsai-lecture-notes/main/notebooks/img/IMPERIAL_logo_RGB_Blue_2024.svg" alt="Imperial Logo" width="500"/><br /><br />

Accessing and Exploring Clinical Data — Solutions
==============
### SURG70098 - Surgical Data Science and AI
### Stuart Bowyer

*Worked solutions to the **» Investigate**, **» Modify** and **» Make** tasks in **03_Accessing_and_Exploring_Clinical_Data**, in the same order. Run the setup cell first; each part then builds on the one before. The counts quoted are for the course views, and the `LIMIT` counts can differ between runs.*

## Setup

In [ ]:
%pip install pandas_gbq --quiet
import pandas as pd
import matplotlib.pyplot as plt
import pandas_gbq

PROJECT = 'bowyerlab-mimic-teaching'

def query(sql, project=PROJECT):
    """Run a SQL query on BigQuery and return the result as a DataFrame."""
    return pandas_gbq.read_gbq(sql, project_id=project)

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.diagnoses_icd
df_diagnoses = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.diagnoses_icd`")

## Querying MIMIC from Python

### What `query` returns
**A pandas DataFrame**, the same kind of table as in Session 2, with **100 rows** (the `LIMIT`) and **6 columns**: `subject_id`, `gender`, `anchor_age`, `anchor_year`, `anchor_year_group` and `dod`. `df_first.shape` gives both numbers. **`subject_id`** identifies the patient, and it appears in every MIMIC table about a patient, so it is the key for joining them: admissions add `hadm_id` for each hospital stay, and ICU tables add `stay_id`.

### The oldest age and the years
**The oldest `anchor_age` is 91.** MIMIC removes the true age of anyone over 89, an identifier in a small group, and records them all as 91. So 91 means "over 89", and an age column needs care before you average it.

**`anchor_year` runs from 2110 to 2208.** Every patient's dates are shifted by their own random offset into the future. Intervals within a patient are real (length of stay, time between tests); dates between patients are not comparable, and the year means nothing. `anchor_year_group` gives the real three-year window, e.g. `2014 - 2016`.

### Investigate: LIMIT
With `LIMIT 10000` there were about 50 AF diagnoses; with `LIMIT 100000`, about 190. **Neither is the number in MIMIC, nor a known fraction of it.** `LIMIT` returns whichever rows the database reaches first, which depends on how the table is stored: not a random sample, not the first patients, and not guaranteed to be the same rows twice. Scaling the count up assumes the first rows are typical, which nothing guarantees. A sample has to be chosen deliberately, which is what the course views do.

### Investigate: reading a view
1. `WHERE itemid = 220045` keeps only heart rate.
2. `AND hadm_id IN (SELECT hadm_id FROM ...cohort)` keeps only the admissions in the cohort view: a second `SELECT`, inside the first.
3. BigQuery charges by the data it reads, and it reads whole columns: every one of the 433 million rows, for each column named. Four columns instead of all of them makes each load cheaper and the result smaller. `SELECT *` is fine on the small tables.

## Part 1: Who is in the cohort?

### Modify: load admissions
Change the view name and the variable: the query is otherwise identical.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.patients
df_patients = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.patients`")

# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.admissions
df_admissions = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.admissions`")
print(len(df_admissions))   # 8647

### Note: column types
**`dod` is a date type** (`dbdate`), and **`admittime` and `dischtime` are `datetime64`**: a date, and a date with a time. A CSV is text, so pandas guesses every column and leaves dates as text unless told otherwise. BigQuery stores a type for each column, and `pandas_gbq` keeps it, so date arithmetic works straight away.

In [ ]:
print(df_patients.dtypes)
print(df_admissions.dtypes)

### Modify: gender
Change the column; `normalize=True` divides each count by the total. About 56% of the cohort are male.

In [ ]:
df_patients['gender'].value_counts(normalize=True)

### Make: plot the gender split

In [ ]:
(df_patients['gender'].value_counts(normalize=True) * 100).plot(kind='bar')
plt.title('ICU cohort by gender')
plt.xlabel('Gender')
plt.ylabel('Patients (%)')
plt.show()

## Part 2: Atrial fibrillation

### How diagnoses link
* **To a patient:** `subject_id`, as in `patients`.
* **To an admission, and so to a time:** `hadm_id`. The diagnosis has no time of its own: its time is the admission's, from `admittime` and `dischtime` in `admissions`.
* **Coding:** `icd_code` holds the code without its dot, and `icd_version` says which system it is from, 9 or 10. The same text can mean different things in the two systems, so a code only means something with its version. `seq_num` is the code's priority within the admission.

### Two coding systems
About 67,800 diagnoses are ICD-9 and 89,100 ICD-10. MIMIC spans the US switch from ICD-9 to ICD-10 in 2015, so admissions before it are coded in ICD-9 and those after in ICD-10. **Counting `42731` alone misses every AF diagnosis made after the switch**: in this cohort, more than half of them.

### Modify: ICD-10 as well
From 1,140 to 2,613 diagnoses. (No ICD-10 code starts with `I48` except AF and flutter, and no ICD-9 code starts with it at all, so `startswith` is safe here; check that for any prefix you use.)

In [ ]:
df_af = df_diagnoses[(df_diagnoses['icd_code'] == '42731') | (df_diagnoses['icd_code'].str.startswith('I48'))]
len(df_af)

### Make: patients with AF
About 1,950 patients, against 2,613 diagnoses. **A patient with AF is usually coded with it at every admission**, so diagnoses outnumber patients. Whether you count diagnoses, admissions or patients changes the answer, and the question decides which is right.

In [ ]:
df_af['subject_id'].nunique()

### Make: gender of patients with AF
About 59% male, against 56% in the whole cohort.

In [ ]:
df_af_patients = df_patients[df_patients['subject_id'].isin(df_af['subject_id'])]
df_af_patients['gender'].value_counts(normalize=True)

## Part 3: A sepsis cohort

### Make: the cohort definition
This uses every code in the table: about 1,430 admissions. Whatever you choose must include both coding systems: the two ICD-9 codes in last year's version of this challenge, `99591` and `78552`, give about 450, because they miss every admission coded in ICD-10. That is a mistake, not a definition.

The real choice is which codes count as sepsis. Keeping only severe sepsis and septic shock (`99592`, `78552`, `R6520`, `R6521`) gives about 1,110 admissions: sepsis with organ dysfunction or shock. Septic shock alone (`78552`, `R6521`) gives about 880. None of these is wrong; what matters is that you state your definition and can say why.

In [ ]:
# My sepsis definition: any ICD-9 or ICD-10 code for sepsis, severe sepsis or septic shock
sepsis_codes = ['99591', '99592', '78552', 'R6520', 'R6521']
is_sepsis = df_diagnoses['icd_code'].isin(sepsis_codes) | df_diagnoses['icd_code'].str.startswith('A41')
df_sepsis = df_diagnoses[is_sepsis][['subject_id', 'hadm_id']].drop_duplicates()
len(df_sepsis)

### Make: their admissions
Merging on both keys keeps one copy of each, and the row count does not change: every sepsis admission is in `admissions`.

In [ ]:
df_sepsis_admissions = df_sepsis.merge(df_admissions, on=['subject_id', 'hadm_id'])
len(df_sepsis_admissions)

### Make: gender of the sepsis cohort
About 56% male: no different from the whole cohort.

In [ ]:
df_sepsis_patients = df_sepsis_admissions.merge(df_patients, on='subject_id')
df_sepsis_patients['gender'].value_counts(normalize=True)

### Make: length of stay
A median of about 10.6 days, with a long tail past 100. The median describes this better than the mean, which the tail pulls up.

In [ ]:
df_sepsis_admissions['los_days'] = (
    (df_sepsis_admissions['dischtime'] - df_sepsis_admissions['admittime']).dt.total_seconds() / (60 * 60 * 24)
)
print(df_sepsis_admissions['los_days'].median())

df_sepsis_admissions['los_days'].hist(bins=50)
plt.xlim(0, 60)
plt.title('Hospital length of stay, sepsis admissions (zoomed to 60 days)')
plt.xlabel('Length of stay (days)')
plt.ylabel('Admissions')
plt.show()

### Note: itemid 220045
**Heart Rate**, in **bpm**, in the category *Routine Vital Signs*. `chartevents` holds every charted measurement in one table, identified only by `itemid`; `d_items` is the dictionary that says what each number means. Raw MIMIC is full of these lookups.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.d_items
df_items = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.d_items`")
df_items[df_items['itemid'] == 220045]

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_icu.chartevents, keeping itemid 220045
df_hr = query("SELECT * FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.chartevents_hr`")

### Make: one admission's heart rate
The admission with the most readings has over 3,000, from a very long ICU stay.

In [ ]:
df_sepsis_hr = df_hr[df_hr['hadm_id'].isin(df_sepsis_admissions['hadm_id'])]
longest = df_sepsis_hr['hadm_id'].value_counts().index[0]

df_one = df_sepsis_hr[df_sepsis_hr['hadm_id'] == longest].sort_values('charttime')
df_one.plot(x='charttime', y='valuenum', legend=False)
plt.title('Heart rate through one sepsis admission')
plt.xlabel('Chart time (shifted date)')
plt.ylabel('Heart rate (bpm)')
plt.show()

### Make: heart rate across the cohort
The first histogram is one bar, because **the largest value is over five million**: a data entry error, not a heart rate. A handful of such values stretch the axis until everything else falls in one bin. `describe()` shows it at once.

Here the limits are 20 to 250 bpm: outside that, a value is far more likely an error than a measurement. Any similar, stated choice is fine; what matters is that you looked, chose, and said so. Session 4 covers cleaning properly. The cleaned distribution centres around 90 bpm, at the top of the normal adult range.

In [ ]:
print(df_sepsis_hr['valuenum'].describe())

plausible = df_sepsis_hr[(df_sepsis_hr['valuenum'] >= 20) & (df_sepsis_hr['valuenum'] <= 250)]
plausible['valuenum'].hist(bins=50)
plt.title('Heart rate readings, sepsis admissions (20-250 bpm)')
plt.xlabel('Heart rate (bpm)')
plt.ylabel('Readings')
plt.show()

### Make (stretch): is sepsis deadly here?
**About 29% of the sepsis admissions ended in death in hospital, against about 8% for the rest of the cohort**: three and a half times as high.

The definition changes the rate as well as the count. Severe sepsis or septic shock only: about 32% died in hospital. Septic shock only: about 34%. A stricter definition selects sicker patients, so a study's mortality figure depends on which definition it used.

Two cautions before reading it as "sepsis kills": the comparison does not account for anything else about the patients (age, other illness), and the diagnosis codes are assigned at discharge, when the outcome is already known. Session 4 builds models of exactly this outcome.

In [ ]:
print(df_sepsis_admissions['hospital_expire_flag'].mean())    # about 0.29

df_other = df_admissions[~df_admissions['hadm_id'].isin(df_sepsis['hadm_id'])]
print(df_other['hospital_expire_flag'].mean())                # about 0.08

### Make (stretch): sepsis from blood cultures
1. About 3,690 of the 8,647 cohort admissions had a blood culture: over 40%. Suspected infection is common in intensive care.
2. Most of the ICD sepsis admissions had a blood culture, but most blood culture admissions have no sepsis code.
3. **Neither is sepsis on its own.** A blood culture shows that infection was suspected, not confirmed, so it is too broad; ICD codes depend on billing practice and on the coding system, so they miss cases and vary over time. Published definitions combine evidence of infection with evidence of organ dysfunction (Sepsis-3: suspected infection with a rise in SOFA score), and a study should say which it used and why.

In [ ]:
# Full MIMIC-IV: physionet-data.mimiciv_3_1_hosp.microbiologyevents (no course view)
df_cultures = query("""
SELECT *
FROM `physionet-data.mimiciv_3_1_hosp.microbiologyevents`
WHERE spec_type_desc = 'BLOOD CULTURE'
  AND hadm_id IN (SELECT hadm_id FROM `bowyerlab-mimic-teaching.sdsai_mimiciv_icu_small_views.cohort`)
""")

culture_admissions = df_cultures['hadm_id'].unique()
print(len(culture_admissions))

sepsis_with_culture = df_sepsis['hadm_id'].isin(culture_admissions)
print(sepsis_with_culture.sum(), 'of', len(df_sepsis), 'sepsis admissions had a blood culture')

culture_without_sepsis = ~pd.Series(culture_admissions).isin(df_sepsis['hadm_id'])
print(culture_without_sepsis.sum(), 'of', len(culture_admissions), 'blood culture admissions have no sepsis code')